<a href="https://colab.research.google.com/github/songhengchang88-commits/Polymarket-Quant-Platform/blob/main/notebooks/Polymarket_Quant_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🌌 Polymarket Quant Platform // Colab 云端深度分析与实时监控大屏

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/songhengchang88-commits/Polymarket-Quant-Platform/blob/main/notebooks/Polymarket_Quant_Colab.ipynb)

---
### 核心能力与架构：
1. **📡 实时交易数据流与多通道监控**：支持公网直连 Polymarket 官方 Gamma/CLOB API，或穿透对接本地量化引擎；
2. **📊 盘口深度与科学计算**：利用 Colab 免费云端算力，计算全网点差分布 (Spread Distribution)、Markout 胜率与蒙特卡洛资金模拟；
3. **🤖 Gemini Pro 智能量化投顾**：集成 Google Gemini 1.5 Pro，输入实时深度盘口切片，全自动输出华尔街级风控诊断与黑天鹅预警。


In [1]:
# 1. 安装核心科学计算、可视化与 Gemini AI 依赖
!pip install -q plotly pandas numpy requests google-generativeai
print('✅ 科学计算、交互图表与 Gemini SDK 依赖安装完毕！')


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-cloud-bigquery-connection 1.23.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-firestore 2.34.1 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-translate 3.28.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-dataproc 5.32.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
tensorboard 2.21.0 requires protobuf<8.0.0,>=6.31.1, but you have protobuf 5.29.6 which is incompatible.
google-cloud-spanner 3.71.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-cloud-resource-manager 1.19.0 requires protobuf<8.0.0,>=6.33.5, but you have protobuf 5.29.6 which is incompatible.
google-clo

## 📡 第一部分：实时交易数据流与多通道监控
- **公网云端直连模式 (全球可用)**：直接从 Polymarket 官方 API 抓取成交量前 50 的主力预测市场，计算配对成本与利差；
- **本地节点穿透模式**：若本地启动了 `ngrok http 8888`，填入 URL 即可实时对账 6.98U 实盘账本与看门狗状态。


In [2]:
import requests
import pandas as pd
import numpy as np
import time
import json
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output, HTML

# 配置监控数据源
LOCAL_TUNNEL_URL = ''  # @param {type:'string'}
GAMMA_API_URL = 'https://gamma-api.polymarket.com/markets?limit=50&active=true&closed=false&order=volume24hr&ascending=false'

def fetch_live_data():
    """优先从本地穿透节点获取实盘账本，否则从官方公网拉取 50 大活跃市场"""
    if LOCAL_TUNNEL_URL:
        try:
            url = f"{LOCAL_TUNNEL_URL.rstrip('/')}/api/status"
            r = requests.get(url, timeout=5)
            if r.status_code == 200:
                return 'local', r.json()
        except Exception as e:
            print(f'本地节点连接提示: {e}，自动回退到公网实时行情')

    try:
        resp = requests.get(GAMMA_API_URL, timeout=8)
        if resp.status_code == 200:
            markets = resp.json()
            rows = []
            for m in markets:
                prices = json.loads(m.get('outcomePrices', '[]')) if isinstance(m.get('outcomePrices'), str) else m.get('outcomePrices', [])
                if len(prices) >= 2:
                    try:
                        p0 = float(prices[0])
                        p1 = float(prices[1])
                        pair_cost = p0 + p1
                        rows.append({
                            'id': str(m.get('id', '')),
                            'question': str(m.get('question', ''))[:40],
                            'volume_24h': float(m.get('volume24hr', 0)),
                            'price_0': p0,
                            'price_1': p1,
                            'pair_cost': round(pair_sum := p0 + p1, 4),
                            'taker_edge': round(1.0 - pair_sum, 4),
                            'safe_range': (0.36 <= p0 <= 0.64) and (0.36 <= p1 <= 0.64),
                        })
                    except Exception:
                        continue
            return 'gamma', pd.DataFrame(rows)
    except Exception as e:
        print(f'公网抓取异常: {e}')
    return 'empty', pd.DataFrame()

print('✅ 数据抓取引擎初始化完成！')


✅ 数据抓取引擎初始化完成！


In [3]:
# 运行动态实时监控大屏 (连续运行 5 轮，每轮间隔 3 秒)
for iteration in range(5):
    mode, data = fetch_live_data()
    clear_output(wait=True)

    if mode == 'local':
        acc = data.get('account', {})
        live = data.get('live_status', {})
        display(HTML(f'''
        <div style="background:#030712; padding:16px; border-radius:12px; color:#e2e8f0; font-family:monospace; border:1px solid #00f3ff;">
            <h3 style="margin:0; color:#00f3ff;">⚡ POLYMARKET QUANT TERMINAL // 本地实盘单一真源对账</h3>
            <p style="color:#94a3b8; font-size:12px;">更新时间: {time.strftime('%Y-%m-%d %H:%M:%S')} | 模式: LIVE 实盘</p>
            <div style="display:flex; gap:16px; margin-top:10px;">
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">链上实盘真实可用</div>
                    <div style="font-size:22px; font-weight:bold; color:#f43f5e;">${acc.get('current_cash', 6.98)} USDC</div>
                </div>
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">45% 仓位硬锁上限</div>
                    <div style="font-size:22px; font-weight:bold; color:#38bdf8;">${live.get('max_order_usd', 3.14)}U</div>
                </div>
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">真实成交流水</div>
                    <div style="font-size:22px; font-weight:bold; color:#4ade80;">{acc.get('trades_count', 0)} 笔 (胜率 {acc.get('win_rate_pct', 0.0)}%)</div>
                </div>
            </div>
        </div>
        '''))
    elif mode == 'gamma' and isinstance(data, pd.DataFrame) and not data.empty:
        df = data
        valid_opps = df[df['pair_cost'] <= 0.985]
        in_budget_zone = df[df['safe_range'] == True]
        min_cost = df['pair_cost'].min()

        display(HTML(f'''
        <div style="background:#030712; padding:16px; border-radius:12px; color:#e2e8f0; font-family:monospace; border:1px solid #00f3ff;">
            <h3 style="margin:0; color:#00f3ff;">🌌 POLYMARKET QUANT TERMINAL // COLAB 云端实时大屏</h3>
            <p style="color:#94a3b8; font-size:12px;">时间: {time.strftime('%Y-%m-%d %H:%M:%S')} | 轮次: {iteration+1}/5 | 数据源: 官方公网实时行情</p>
            <div style="display:flex; gap:16px; margin-top:10px;">
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">实时扫描盘口</div>
                    <div style="font-size:20px; font-weight:bold; color:#38bdf8;">{len(df)} 个</div>
                </div>
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">Taker 利差机会 (&le;0.985)</div>
                    <div style="font-size:20px; font-weight:bold; color:{'#4ade80' if len(valid_opps)>0 else '#f87171'};
                    ">{len(valid_opps)} 个</div>
                </div>
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">6.98U 可买区间 [0.36~0.64]</div>
                    <div style="font-size:20px; font-weight:bold; color:#fbbf24;">{len(in_budget_zone)} 个</div>
                </div>
                <div style="background:#0f172a; padding:10px 16px; border-radius:8px;">
                    <div style="font-size:11px; color:#64748b;">全网最低配对成本</div>
                    <div style="font-size:20px; font-weight:bold; color:#a78bfa;">{min_cost:.4f}</div>
                </div>
            </div>
        </div>
        '''))

        # 绘制实时点差柱状图
        top_df = df.head(15)
        fig = go.Figure()
        fig.add_trace(go.Bar(
            x=top_df['question'],
            y=top_df['pair_cost'],
            marker_color=np.where(top_df['pair_cost'] <= 0.985, '#4ade80', '#00f3ff'),
            name='配对买入成本 (Ask0 + Ask1)'
        ))
        fig.add_hline(y=0.985, line_dash='dash', line_color='#f43f5e', annotation_text='0.985 Taker 红线')
        fig.update_layout(
            title='Top 15 活跃市场实时配对成本雷达',
            template='plotly_dark',
            yaxis=dict(range=[0.94, 1.03], title='配对总成本'),
            xaxis=dict(tickangle=35),
            height=400,
            margin=dict(l=40, r=40, t=50, b=120)
        )
        fig.show()
    time.sleep(3)


## 🔬 第二部分：科学计算与 6.98U 蒙特卡洛资金演化模拟
在坚守 **45% 仓位硬锁 (3.14U)**、**单腿 $1.15 底线** 与 **FOK 零单腿裸持** 铁律下：
- 假设每笔成功配对净利：`+0.035 USDC`
- 假设发生单腿滑点亏损：`-0.15 USDC` (一次亏损抵消 4~5 笔盈利)
以下推演 50 条随机演化轨迹，科学揭示资金曲线的数学客观规律。


In [4]:
import numpy as np
import plotly.graph_objects as go

np.random.seed(42)
INITIAL_CASH = 6.98
TRIALS = 50
STEPS = 80
WIN_RATE = 0.95  # 95% 高胜率假设
WIN_PAYOUT = 0.035
LOSS_PER_FAILURE = 0.15

paths = np.zeros((TRIALS, STEPS))
paths[:, 0] = INITIAL_CASH

for t in range(TRIALS):
    for s in range(1, STEPS):
        cur = paths[t, s-1]
        if cur <= 2.30:  # 资金低于两腿底线 (2 x 1.15U)，无法再开仓
            paths[t, s] = cur
            continue
        is_win = np.random.rand() < WIN_RATE
        change = WIN_PAYOUT if is_win else -LOSS_PER_FAILURE
        paths[t, s] = max(0.0, cur + change)

fig = go.Figure()
for t in range(TRIALS):
    fig.add_trace(go.Scatter(y=paths[t], mode='lines', line=dict(width=1), opacity=0.35, showlegend=False))

fig.add_trace(go.Scatter(y=np.median(paths, axis=0), mode='lines', line=dict(color='#00f3ff', width=3), name='中位数稳健轨迹'))
fig.add_hline(y=INITIAL_CASH, line_dash='dot', line_color='#fbbf24', annotation_text='初始资金 6.98U')
fig.add_hline(y=2.30, line_dash='dash', line_color='#f43f5e', annotation_text='最小开仓红线 2.30U')

fig.update_layout(
    title='6.98U 本金蒙特卡洛资金模拟 (50条演化路径)',
    template='plotly_dark',
    xaxis_title='交易笔数 (Trades)',
    yaxis_title='账户资产 (USDC)',
    height=420,
)
fig.show()


## 🤖 第三部分：Google Gemini Pro 智能量化研判
利用 Colab 原生集成的 Google Gemini 1.5 Pro 模型，对最新实时盘口数据做华尔街级风控研判。


In [5]:
import google.generativeai as genai

# 免费从 https://aistudio.google.com 获取 API Key 填入下方
GEMINI_API_KEY = ''  # @param {type:'string'}

if GEMINI_API_KEY:
    genai.configure(api_key=GEMINI_API_KEY)
    model = genai.GenerativeModel('gemini-1.5-pro-latest')

    # 获取最新 5 个盘口切片
    _, df_markets = fetch_live_data()
    if isinstance(df_markets, pd.DataFrame) and not df_markets.empty:
        slice_str = df_markets.head(5)[['question', 'pair_cost', 'volume_24h', 'safe_range']].to_string()

        prompt = f"""
        你是一位顶级高频量化与预测市场首席风控官。
        当前我们的实盘账户有 6.98 USDC 防御资金，仓位硬锁 45% (3.14U)，单腿底线 1.15U，执行零裸持铁律。
        以下是 Polymarket 最新的 5 个主力市场深度盘口快照：
        {slice_str}

        请简明扼要给出专业诊断：
        1. 盘口质量与是否存在真实无风险利差 (ask0+ask1 <= 0.985)？
        2. 6.98U 本金在这些市场中的可操作性诊断 (为何大选两极化盘口无法进场)？
        3. 给当前阶段的最高风控指令。
        """
        print('🤖 正在调用 Gemini 1.5 Pro 进行深度量化研判...')
        resp = model.generate_content(prompt)
        display(HTML(f'''
        <div style="background:#064e3b; padding:16px; border-radius:10px; border:1px solid #10b981; color:#ecfdf5; font-family:sans-serif;">
            <h4 style="margin:0 0 10px 0; color:#34d399;">✅ Gemini 1.5 Pro 量化风控研判报告</h4>
            <div style="white-space: pre-wrap; line-height:1.6;">{resp.text}</div>
        </div>
        '''))
else:
    print('💡 提示: 填入 GEMINI_API_KEY 即可解锁 Gemini 1.5 Pro 全自动智能研判！')


💡 提示: 填入 GEMINI_API_KEY 即可解锁 Gemini 1.5 Pro 全自动智能研判！


/tmp/ipykernel_349/504818828.py:1: FutureWarning:



All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md


